# Visual Fingerprinting Methods: pHash vs dHash vs HSV Color Histogram

## What is being compared?
Three visual fingerprinting methods are compared here:

- **pHash (Perceptual Hash)**: Converts a frame to grayscale, resizes it, applies a Discrete Cosine Transform (DCT), and encodes the low-frequency coefficients into a compact hash. Similar images produce similar hashes.
- **dHash (Difference Hash)**: Converts a frame to grayscale, resizes it, and encodes the gradient direction between adjacent pixels into a hash. Captures edge and structural information.
- **HSV Color Histogram**: Converts a frame to HSV color space and computes per-channel histograms. Captures the color distribution of the frame rather than structural content.

## Why?
pHash and dHash are both compact and fast to compare, but sensitive to different visual features. Color histograms capture color distribution which may be more robust to certain edits (cropping, overlays) but are more sensitive to color grading changes. This notebook visualizes the processing pipeline for each method and generates fingerprints for all videos using a chunked extraction strategy — 4-second chunks, 1-second overlap, 5fps — to evaluate which method best captures visual similarity for later comparison.

## Chunking Strategy
Each video is divided into 4-second chunks with a 1-second overlap between consecutive chunks. Within each chunk, frames are sampled at 5fps using `cap.get` for accurate fixed-rate extraction. This produces 20 frames per chunk and ensures temporal coverage without redundancy.

In [ ]:
import cv2
import numpy as np
import imagehash
import os
import json
import time
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial.distance import cosine as cosine_dist

# Config
MARGIN = 30
HASH_SIZE = 8
CHUNK_DURATION_SEC = 4
OVERLAP_SEC = 1
DESIRED_FPS = 5
HIST_BINS = 32
OUTPUT_JSON = "visual_fingerprints.json"
SAMPLE_VIDEO = "../data/downloads/scottkress_halloween_dl.mp4"  # Update to any available video

FOLDERS_TO_PROCESS = [
    "../data/downloads",
    "../data/processed/reels_processed",
    "../data/processed/shorts_processed"
]

# Update these to match your actual video filenames as they appear in the JSON

TARGET_VIDEO = "scottkress_halloween_dl.mp4"
POSITIVE_VIDEOS = [
    "scottkress_halloween_ig.mp4",
    "scottkress_halloween_yt.mp4"
]
TOP_K = 5  # Number of top chunk matches to average for mean top-k score

In [ ]:
# --- Shared frame helpers ---

def get_nth_frame(video_path, frame_index=50):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise Exception(f"Could not open video {video_path}")
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise Exception(f"Could not read frame {frame_index} from {video_path}")
    return frame

def crop_frame(frame, margin=MARGIN):
    h, w = frame.shape[:2]
    return frame[margin:h-margin, margin:w-margin]

# --- Chunking logic ---

def get_chunks(video_path, chunk_duration=CHUNK_DURATION_SEC, overlap=OVERLAP_SEC, desired_fps=DESIRED_FPS):
    """
    Returns a list of chunks, each containing sampled frames.
    Each chunk: {chunk_id, start_sec, end_sec, frames: [frame, ...]}
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return []

    native_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total_frames / native_fps

    step_sec = chunk_duration - overlap
    frame_interval = max(int(native_fps / desired_fps), 1)

    chunks = []
    chunk_id = 0
    start_sec = 0.0

    while start_sec < duration_sec:
        end_sec = min(start_sec + chunk_duration, duration_sec)
        start_frame = int(start_sec * native_fps)
        end_frame = int(end_sec * native_fps)

        frames = []
        for f_idx in range(start_frame, end_frame, frame_interval):
            cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
            ret, frame = cap.read()
            if ret:
                frames.append((f_idx, frame))

        chunks.append({
            "chunk_id": chunk_id,
            "start_sec": round(start_sec, 3),
            "end_sec": round(end_sec, 3),
            "frames": frames
        })

        chunk_id += 1
        start_sec += step_sec

    cap.release()
    return chunks

## Visual Representations

In [ ]:
# --- pHash pipeline visualization ---

def phash_pipeline(frame):
    steps = {}
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    steps["Original"] = (frame_rgb, None)

    cropped = crop_frame(frame)
    steps[f"Cropped ({MARGIN}px)"] = (cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB), None)

    gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
    steps["Grayscale"] = (gray, "gray")

    resized = cv2.resize(gray, (HASH_SIZE * 4, HASH_SIZE * 4))
    steps["Resized (DCT Input)"] = (resized, "gray")

    dct = cv2.dct(np.float32(resized))
    steps["DCT Spectrum"] = (np.log(np.abs(dct) + 1), "inferno")

    pil_img = Image.fromarray(gray)
    phash = imagehash.phash(pil_img)
    steps["pHash 8x8 Grid"] = (np.array(phash.hash, dtype=int), "gray")

    return steps, phash

frame = get_nth_frame(SAMPLE_VIDEO, frame_index=50)
steps, phash = phash_pipeline(frame)

fig, axes = plt.subplots(1, len(steps), figsize=(3 * len(steps), 4))
for ax, (title, (img, cmap)) in zip(axes, steps.items()):
    ax.imshow(img, cmap=cmap, interpolation="nearest")
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.suptitle(f"pHash Pipeline — {os.path.basename(SAMPLE_VIDEO)}\nHash: {phash}", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- dHash pipeline visualization ---

def dhash_pipeline(frame):
    steps = {}
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    steps["Original"] = (frame_rgb, None)

    cropped = crop_frame(frame)
    steps[f"Cropped ({MARGIN}px)"] = (cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB), None)

    gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
    steps["Grayscale"] = (gray, "gray")

    # dHash resizes to (hash_size+1) x hash_size to compute horizontal gradient
    resized = cv2.resize(gray, (HASH_SIZE + 1, HASH_SIZE))
    steps["Resized (Gradient Input)"] = (resized, "gray")

    gradient = resized[:, 1:] > resized[:, :-1]
    steps["Horizontal Gradient"] = (gradient.astype(np.uint8) * 255, "gray")

    pil_img = Image.fromarray(gray)
    dhash = imagehash.dhash(pil_img)
    steps["dHash 8x8 Grid"] = (np.array(dhash.hash, dtype=int), "gray")

    return steps, dhash

frame = get_nth_frame(SAMPLE_VIDEO, frame_index=50)
steps, dhash = dhash_pipeline(frame)

fig, axes = plt.subplots(1, len(steps), figsize=(3 * len(steps), 4))
for ax, (title, (img, cmap)) in zip(axes, steps.items()):
    ax.imshow(img, cmap=cmap, interpolation="nearest")
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.suptitle(f"dHash Pipeline — {os.path.basename(SAMPLE_VIDEO)}\nHash: {dhash}", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- HSV Color Histogram pipeline visualization ---

def hsv_histogram_pipeline(frame):
    steps = {}
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    steps["Original"] = (frame_rgb, None)

    cropped = crop_frame(frame)
    steps[f"Cropped ({MARGIN}px)"] = (cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB), None)

    hsv = cv2.cvtColor(cropped, cv2.COLOR_BGR2HSV)
    steps["HSV"] = (hsv, None)

    h_hist = cv2.calcHist([hsv], [0], None, [HIST_BINS], [0, 180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], None, [HIST_BINS], [0, 256]).flatten()
    v_hist = cv2.calcHist([hsv], [2], None, [HIST_BINS], [0, 256]).flatten()

    # Normalize
    h_hist /= (h_hist.sum() + 1e-7)
    s_hist /= (s_hist.sum() + 1e-7)
    v_hist /= (v_hist.sum() + 1e-7)

    histogram = np.concatenate([h_hist, s_hist, v_hist])
    return steps, h_hist, s_hist, v_hist, histogram

frame = get_nth_frame(SAMPLE_VIDEO, frame_index=50)
steps, h_hist, s_hist, v_hist, histogram = hsv_histogram_pipeline(frame)

fig = plt.figure(figsize=(20, 4))

# Pipeline images
for i, (title, (img, cmap)) in enumerate(steps.items()):
    ax = fig.add_subplot(1, len(steps) + 3, i + 1)
    ax.imshow(img, cmap=cmap, interpolation="nearest")
    ax.set_title(title, fontsize=9)
    ax.axis("off")

# Per-channel histograms
offset = len(steps) + 1
for i, (hist, label, color) in enumerate(zip(
    [h_hist, s_hist, v_hist],
    ["Hue", "Saturation", "Value"],
    ["r", "g", "b"]
)):
    ax = fig.add_subplot(1, len(steps) + 3, offset + i)
    ax.bar(range(len(hist)), hist, color=color, alpha=0.7)
    ax.set_title(f"{label} Histogram", fontsize=9)
    ax.set_xlabel("Bin")
    ax.set_ylabel("Normalized Freq")

plt.suptitle(f"HSV Histogram Pipeline — {os.path.basename(SAMPLE_VIDEO)}", fontsize=11)
plt.tight_layout()
plt.show()

## Computation and Encodings

In [ ]:
# --- Generate and save all methods to JSON ---

def compute_phash(frame):
    cropped = crop_frame(frame)
    gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
    return str(imagehash.phash(Image.fromarray(gray)))

def compute_dhash(frame):
    cropped = crop_frame(frame)
    gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
    return str(imagehash.dhash(Image.fromarray(gray)))

def compute_hsv_histogram(frame):
    cropped = crop_frame(frame)
    hsv = cv2.cvtColor(cropped, cv2.COLOR_BGR2HSV)
    h_hist = cv2.calcHist([hsv], [0], None, [HIST_BINS], [0, 180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], None, [HIST_BINS], [0, 256]).flatten()
    v_hist = cv2.calcHist([hsv], [2], None, [HIST_BINS], [0, 256]).flatten()
    h_hist /= (h_hist.sum() + 1e-7)
    s_hist /= (s_hist.sum() + 1e-7)
    v_hist /= (v_hist.sum() + 1e-7)
    return np.concatenate([h_hist, s_hist, v_hist]).tolist()

all_data = {}
timing = {}  # {video_name: {phash: t, dhash: t, hsv: t, total: t}}
total_start = time.time()

for folder in FOLDERS_TO_PROCESS:
    folder_path = os.path.abspath(folder)
    if not os.path.exists(folder_path):
        print(f"Skipping missing folder: {folder_path}")
        continue

    for file_name in os.listdir(folder_path):
        if not file_name.lower().endswith((".mp4", ".mov", ".mkv")):
            continue

        video_path = os.path.join(folder_path, file_name)
        print(f"Processing: {file_name}")

        chunks = get_chunks(video_path)
        video_entry = {"chunks": []}
        timing[file_name] = {"phash": 0.0, "dhash": 0.0, "hsv": 0.0}

        for chunk in chunks:
            chunk_entry = {
                "chunk_id": chunk["chunk_id"],
                "start_sec": chunk["start_sec"],
                "end_sec": chunk["end_sec"],
                "frames": []
            }

            for f_idx, frame in chunk["frames"]:
                t0 = time.time()
                ph = compute_phash(frame)
                timing[file_name]["phash"] += time.time() - t0

                t0 = time.time()
                dh = compute_dhash(frame)
                timing[file_name]["dhash"] += time.time() - t0

                t0 = time.time()
                hsv = compute_hsv_histogram(frame)
                timing[file_name]["hsv"] += time.time() - t0

                chunk_entry["frames"].append({
                    "frame_idx": f_idx,
                    "phash": ph,
                    "dhash": dh,
                    "color_histogram": hsv
                })

            video_entry["chunks"].append(chunk_entry)

        timing[file_name]["total"] = round(
            timing[file_name]["phash"] + timing[file_name]["dhash"] + timing[file_name]["hsv"], 4
        )
        all_data[file_name] = video_entry
        print(f"  Done. Chunks: {len(chunks)} | Time: {timing[file_name]['total']}s")

total_elapsed = round(time.time() - total_start, 3)

with open(OUTPUT_JSON, "w") as f:
    json.dump(all_data, f)

print(f"\nJSON saved to {OUTPUT_JSON}")
print(f"Total generation time: {total_elapsed}s")

In [ ]:
# --- Timing and storage metrics ---

json_size_kb = os.path.getsize(OUTPUT_JSON) / 1024

total_phash = sum(v["phash"] for v in timing.values())
total_dhash = sum(v["dhash"] for v in timing.values())
total_hsv   = sum(v["hsv"]   for v in timing.values())

print(f"{'Video':<45} {'pHash (s)':>10} {'dHash (s)':>10} {'HSV (s)':>10} {'Total (s)':>10}")
print("-" * 87)
for video, t in timing.items():
    print(f"{video:<45} {t['phash']:>10.4f} {t['dhash']:>10.4f} {t['hsv']:>10.4f} {t['total']:>10.4f}")
print("-" * 87)
print(f"{'TOTAL':<45} {total_phash:>10.4f} {total_dhash:>10.4f} {total_hsv:>10.4f} {total_elapsed:>10.4f}")
print(f"\nJSON file size: {json_size_kb:.2f} KB")

In [ ]:
# --- Load JSON and extract chunk-level fingerprints ---

with open(OUTPUT_JSON, "r") as f:
    fingerprint_data = json.load(f)

def get_chunk_fingerprints(video_name, method):
    """
    Returns a list of per-chunk aggregated fingerprints for a given method.
    For hash methods: list of imagehash objects (one representative per chunk, majority or first).
    For hsv: list of mean histogram vectors per chunk.
    """
    chunks = fingerprint_data[video_name]["chunks"]
    result = []
    for chunk in chunks:
        frames = chunk["frames"]
        if not frames:
            continue
        if method in ("phash", "dhash"):
            # Use first frame hash as chunk representative
            result.append(imagehash.hex_to_hash(frames[0][method]))
        elif method == "color_histogram":
            vecs = np.array([f["color_histogram"] for f in frames])
            result.append(np.mean(vecs, axis=0))
    return result

def chunk_distance(c1, c2, method):
    """Compute distance between two chunk fingerprints."""
    if method in ("phash", "dhash"):
        return abs(c1 - c2)
    elif method == "color_histogram":
        return cosine_dist(c1, c2)

def compare_videos(target_chunks, other_chunks, method, top_k=TOP_K):
    """
    All-vs-all chunk comparison.
    Returns min_diff and mean of top-k best matching pairs.
    """
    all_dists = []
    for c1 in target_chunks:
        for c2 in other_chunks:
            all_dists.append(chunk_distance(c1, c2, method))

    if not all_dists:
        return None, None

    all_dists_sorted = sorted(all_dists)
    min_diff = all_dists_sorted[0]
    mean_top_k = np.mean(all_dists_sorted[:top_k])
    return min_diff, mean_top_k

print("Comparison functions ready.")

In [ ]:
# --- Run comparisons and compute dynamic thresholds ---

METHODS = ["phash", "dhash", "color_histogram"]
all_videos = list(fingerprint_data.keys())
target_chunks = {m: get_chunk_fingerprints(TARGET_VIDEO, m) for m in METHODS}

# Store all results: {method: {video: {min_diff, mean_top_k}}}
results = {m: {} for m in METHODS}

for method in METHODS:
    for video in all_videos:
        if video == TARGET_VIDEO:
            continue
        other_chunks = get_chunk_fingerprints(video, method)
        min_diff, mean_top_k = compare_videos(target_chunks[method], other_chunks, method)
        results[method][video] = {"min_diff": min_diff, "mean_top_k": mean_top_k}

# Dynamic thresholds: worst (highest) score among positive videos
thresholds = {}
for method in METHODS:
    pos_min_diffs = [results[method][v]["min_diff"] for v in POSITIVE_VIDEOS if v in results[method]]
    pos_mean_top_k = [results[method][v]["mean_top_k"] for v in POSITIVE_VIDEOS if v in results[method]]
    thresholds[method] = {
        "min_diff": max(pos_min_diffs),
        "mean_top_k": max(pos_mean_top_k)
    }

print(f"{'Method':<20} {'min_diff threshold':>20} {'mean_top_k threshold':>22}")
print("-" * 64)
for method, t in thresholds.items():
    print(f"{method:<20} {t['min_diff']:>20.4f} {t['mean_top_k']:>22.4f}")

In [ ]:
# --- Confusion matrices per method ---

def build_confusion(results_for_method, positive_videos, threshold_min_diff, threshold_mean_top_k):
    """
    Classifies each video as match/no-match using each threshold.
    Returns two confusion dicts (one per score type):
      {video: {label, predicted_min_diff, predicted_mean_top_k}}
    """
    rows = []
    for video, scores in results_for_method.items():
        is_positive = video in positive_videos
        pred_min = scores["min_diff"] <= threshold_min_diff
        pred_topk = scores["mean_top_k"] <= threshold_mean_top_k
        rows.append({
            "video": video,
            "actual": is_positive,
            "pred_min_diff": pred_min,
            "pred_mean_top_k": pred_topk,
            "min_diff": scores["min_diff"],
            "mean_top_k": scores["mean_top_k"]
        })
    return rows

def confusion_counts(rows, pred_key):
    tp = sum(1 for r in rows if r["actual"] and r[pred_key])
    fp = sum(1 for r in rows if not r["actual"] and r[pred_key])
    tn = sum(1 for r in rows if not r["actual"] and not r[pred_key])
    fn = sum(1 for r in rows if r["actual"] and not r[pred_key])
    return tp, fp, tn, fn

def plot_confusion_matrix(ax, tp, fp, tn, fn, title):
    matrix = np.array([[tp, fn], [fp, tn]])
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Predicted Match", "Predicted No Match"])
    ax.set_yticklabels(["Actual Positive", "Actual Negative"])
    ax.set_title(title, fontsize=10)
    labels = [[f"TP\n{tp}", f"FN\n{fn}"], [f"FP\n{fp}", f"TN\n{tn}"]]
    for i in range(2):
        for j in range(2):
            ax.text(j, i, labels[i][j], ha="center", va="center", fontsize=12, color="black")

fig, axes = plt.subplots(len(METHODS), 2, figsize=(12, 5 * len(METHODS)))

for row_idx, method in enumerate(METHODS):
    rows = build_confusion(
        results[method], POSITIVE_VIDEOS,
        thresholds[method]["min_diff"],
        thresholds[method]["mean_top_k"]
    )

    tp, fp, tn, fn = confusion_counts(rows, "pred_min_diff")
    plot_confusion_matrix(axes[row_idx, 0], tp, fp, tn, fn,
        f"{method} — min_diff (threshold: {thresholds[method]['min_diff']:.4f})")

    tp, fp, tn, fn = confusion_counts(rows, "pred_mean_top_k")
    plot_confusion_matrix(axes[row_idx, 1], tp, fp, tn, fn,
        f"{method} — mean top-{TOP_K} (threshold: {thresholds[method]['mean_top_k']:.4f})")

plt.suptitle(f"Confusion Matrices — Target: {TARGET_VIDEO}", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Print raw scores for reference
print(f"\n{'Video':<45} {'Method':<20} {'min_diff':>10} {'mean_top_k':>12}")
print("-" * 90)
for method in METHODS:
    for video, scores in sorted(results[method].items(), key=lambda x: x[1]['min_diff']):
        marker = " *" if video in POSITIVE_VIDEOS else ""
        print(f"{video:<45} {method:<20} {scores['min_diff']:>10.4f} {scores['mean_top_k']:>12.4f}{marker}")
print("\n* = positive video")

## Conclusion

**Method chosen:** 

**Reasoning:** 